# Act 2 — Distribution portraits

> **Opening question:** *"If you lined up every house in Ames, Iowa from cheapest to most expensive and took a photo — what shape would that crowd make? Is it a neat bell curve, or do a handful of mansions drag the tail way out to the right?"*

---
Before we ask *why* prices differ, we need to understand *what normal looks like*.
This act profiles the key numeric columns that drive SalePrice.

In [ ]:
import sys
sys.path.append('..')
from src.utils import *

act_header(
    act_num=2,
    title="Distribution portraits",
    opening_question="What does a typical Ames house look like — and how wide is the gap between typical and extraordinary?"
)

df = load_processed('cleaned.csv')

## 2.1 — The price tag distribution
The most important column in the dataset. Let's see its shape before anything else.

In [ ]:
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Raw SalePrice
sns.histplot(df['SalePrice'], kde=True, color=PALETTE['purple'], ax=axes[0], linewidth=0)
axes[0].axvline(df['SalePrice'].mean(),   color=PALETTE['coral'], linestyle='--', lw=1.8, label=f"Mean: ${df['SalePrice'].mean():,.0f}")
axes[0].axvline(df['SalePrice'].median(), color=PALETTE['teal'],  linestyle='--', lw=1.8, label=f"Median: ${df['SalePrice'].median():,.0f}")
axes[0].set_title('Sale price — raw', fontsize=13)
axes[0].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x/1000:.0f}k'))
axes[0].legend(fontsize=10)

# Log-transformed (more normal)
sns.histplot(np.log1p(df['SalePrice']), kde=True, color=PALETTE['teal'], ax=axes[1], linewidth=0)
axes[1].set_title('Sale price — log scale (more balanced view)', fontsize=13)
axes[1].set_xlabel('log(SalePrice)')

plt.suptitle('How are house prices spread across Ames?', fontsize=14, y=1.02)
plt.tight_layout()
save_figure(fig, 'act2_saleprice_distribution.png')
plt.show()

In [ ]:
insight_callout(
    f"The average sale price is ${df['SalePrice'].mean():,.0f} but the median is ${df['SalePrice'].median():,.0f}.\n"
    "When the mean is higher than the median, it means a small number of very expensive\n"
    "homes are pulling the average up — like a few millionaires skewing a neighbourhood's\n"
    "average income. The log-scaled view shows the 'true middle' more honestly.",
    label="Why mean ≠ median here"
)

## 2.2 — Above-ground living area (GrLivArea)
The most intuitive driver of price: how big is the house?

In [ ]:
plot_distribution(df, 'GrLivArea', color=PALETTE['purple'])

insight_callout(
    f"Most homes sit between {df['GrLivArea'].quantile(0.25):,.0f} and {df['GrLivArea'].quantile(0.75):,.0f} sq ft.\n"
    "A handful of outliers exceed 4,000 sq ft — those are the properties we'll investigate in Act 5."
)

## 2.3 — Overall quality score (OverallQual)
Rated 1–10 by assessors. Does quality follow a normal distribution or do assessors cluster around certain scores?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Count plot
qual_counts = df['OverallQual'].value_counts().sort_index()
axes[0].bar(qual_counts.index, qual_counts.values, color=PALETTE['purple'], alpha=0.85, edgecolor='white', linewidth=0.5)
axes[0].set_xlabel('Overall quality score (1=Poor, 10=Excellent)')
axes[0].set_ylabel('Number of houses')
axes[0].set_title('How are quality scores distributed?', fontsize=13)
axes[0].set_xticks(range(1, 11))

# Median price per quality score
price_by_qual = df.groupby('OverallQual')['SalePrice'].median()
axes[1].bar(price_by_qual.index, price_by_qual.values / 1000, color=PALETTE['teal'], alpha=0.85, edgecolor='white', linewidth=0.5)
axes[1].set_xlabel('Overall quality score')
axes[1].set_ylabel('Median sale price ($k)')
axes[1].set_title('Median price at each quality level', fontsize=13)
axes[1].set_xticks(range(1, 11))

plt.tight_layout()
save_figure(fig, 'act2_overallqual.png')
plt.show()

insight_callout(
    "Most houses score between 5 and 7 — assessors rarely give 1s or 10s.\n"
    "But each step up in quality adds significant price: a score-8 house sells for roughly\n"
    f"${price_by_qual.get(8, 0)/1000:.0f}k — nearly double the median score-5 home at ${price_by_qual.get(5, 0)/1000:.0f}k."
)

## 2.4 — Year built vs year remodelled
Age matters — but does a recent remodel make an old house feel new again?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

sns.histplot(df['YearBuilt'],    kde=True, color=PALETTE['blue'],  ax=axes[0], linewidth=0, bins=40)
axes[0].set_title('When were Ames houses built?', fontsize=13)
axes[0].set_xlabel('Year built')

sns.histplot(df['YearRemodAdd'], kde=True, color=PALETTE['amber'], ax=axes[1], linewidth=0, bins=40)
axes[1].set_title('When were they last remodelled?', fontsize=13)
axes[1].set_xlabel('Year remodelled')

plt.tight_layout()
save_figure(fig, 'act2_year_distributions.png')
plt.show()

insight_callout(
    "There are two building booms visible: post-WW2 (1950s) and the 1990s–2000s suburban expansion.\n"
    "The remodel chart spikes near 2000–2007 — just before the financial crisis stalled construction.\n"
    "In Act 4 we'll see exactly how the market responded to that moment."
)

## 2.5 — Key numeric columns at a glance

In [ ]:
key_cols = ['SalePrice', 'GrLivArea', 'TotalBsmtSF', 'GarageArea', 'LotArea']
existing = [c for c in key_cols if c in df.columns]

fig, axes = plt.subplots(1, len(existing), figsize=(14, 4))
colors = list(PALETTE.values())

for i, (col, ax) in enumerate(zip(existing, axes)):
    sns.boxplot(y=df[col], ax=ax, color=colors[i % len(colors)], width=0.5,
                flierprops=dict(marker='o', markerfacecolor=PALETTE['coral'], markersize=3, alpha=0.5))
    ax.set_title(col, fontsize=11)
    ax.set_ylabel('')
    if col == 'SalePrice':
        ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x/1000:.0f}k'))

plt.suptitle('Outlier view — where do extreme values live?', fontsize=13, y=1.01)
plt.tight_layout()
save_figure(fig, 'act2_boxplots_overview.png')
plt.show()

---
## Act 2 — Closing punchline

In [ ]:
punchline(
    "House prices in Ames are right-skewed — a few luxury homes pull the average up. "
    "Most homes are mid-quality (5–7) and mid-size. "
    "In Act 3, we find out which of these variables actually moves the price needle the most."
)


**Next → [Act 3 — Hidden Relationships](03_act3_relationships.ipynb)**